# Last.fm Sessions — Coding Challenge

**Question:** What are the top 10 songs played in the top 50 longest sessions by track count?

**Session definition:** A user session is one or more songs played by a given user where each song starts within **20 minutes** of the previous song's start time.

## Dataset
Download from http://ocelma.net/MusicRecommendationDataset/lastfm-1K.html  
Unzip and upload `userid-timestamp-artid-artname-traid-traname.tsv` to the dashboard landing zone.

## Algorithm
1. Parse the TSV and cast timestamps
2. For each user, compute the gap to the previous play using `lag()` over a per-user time-ordered window
3. Flag rows where the gap exceeds 20 minutes — these are session boundaries
4. Cumsum the flags per user to produce a monotonically increasing session ID
5. Count tracks per session → find top 50 sessions
6. Join back to plays, count songs within those sessions → top 10

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("LastFmSessions")
    .config("spark.sql.shuffle.partitions", "8")  # keep low for local mode
    .getOrCreate()
)

spark.version

## Step 1 — Load and parse the TSV

The dataset has no header. Columns are:
`user_id | timestamp | artist_id | artist_name | track_id | track_name`

In [ ]:
TSV_PATH = "/data/warehouse/landing/userid-timestamp-artid-artname-traid-traname.tsv"

raw = (
    spark.read
    .option("sep", "\t")
    .option("header", "false")
    .option("inferSchema", "false")   # read everything as string first
    .csv(TSV_PATH)
    .toDF("user_id", "ts_raw", "artist_id", "artist_name", "track_id", "track_name")
)

print(f"Raw row count: {raw.count():,}")
raw.show(3, truncate=60)

In [ ]:
# Parse ISO-8601 timestamps and drop rows with null timestamps or track names
plays = (
    raw
    .withColumn("ts", F.to_timestamp("ts_raw", "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .filter(F.col("ts").isNotNull() & F.col("track_name").isNotNull())
    .select("user_id", "ts", "artist_name", "track_name")
)

print(f"Clean rows: {plays.count():,}")
plays.show(3, truncate=50)

## Step 2 — Sessionise

For each user, order plays by timestamp and compute the gap to the previous play.
A new session starts whenever the gap exceeds 20 minutes (1200 seconds).

We create a `session_id` by taking the cumulative sum of session-break flags per user — every time a break is detected the counter increments, giving each session a unique integer within the user.

In [ ]:
SESSION_GAP_SECONDS = 20 * 60  # 20 minutes

user_time_window = Window.partitionBy("user_id").orderBy("ts")

sessionised = (
    plays
    # Time of the previous play for this user
    .withColumn("prev_ts", F.lag("ts").over(user_time_window))
    # Gap in seconds (null on the first play of each user → treat as new session)
    .withColumn(
        "gap_seconds",
        F.unix_timestamp("ts") - F.unix_timestamp("prev_ts")
    )
    # 1 if this play starts a new session, else 0
    .withColumn(
        "is_new_session",
        (F.col("gap_seconds").isNull() | (F.col("gap_seconds") > SESSION_GAP_SECONDS)).cast("int")
    )
    # Cumulative sum gives a monotonically increasing session counter per user
    .withColumn(
        "session_id",
        F.concat_ws(
            "_",
            F.col("user_id"),
            F.sum("is_new_session").over(user_time_window).cast("string")
        )
    )
)

sessionised.select("user_id", "ts", "track_name", "gap_seconds", "is_new_session", "session_id") \
           .show(10, truncate=50)

## Step 3 — Top 50 sessions by track count

In [ ]:
session_sizes = (
    sessionised
    .groupBy("session_id", "user_id")
    .agg(F.count("*").alias("track_count"))
)

top50_sessions = (
    session_sizes
    .orderBy(F.col("track_count").desc())
    .limit(50)
)

print("Top 50 sessions:")
top50_sessions.show(10)

## Step 4 — Top 10 songs within those 50 sessions

In [ ]:
top10_songs = (
    sessionised
    # Keep only plays that belong to one of the top 50 sessions
    .join(top50_sessions.select("session_id"), on="session_id", how="inner")
    # Count how many times each (artist, track) appears across those sessions
    .groupBy("artist_name", "track_name")
    .agg(F.count("*").alias("play_count"))
    .orderBy(F.col("play_count").desc())
    .limit(10)
)

print("Top 10 songs in the top 50 longest sessions:")
top10_songs.show(truncate=60)

## Step 5 — Persist results as Delta tables

Write both results to Delta so you can query them later without re-running the full pipeline.

In [ ]:
top50_sessions.write.format("delta").mode("overwrite") \
    .save("/data/warehouse/delta/lastfm_top50_sessions")

top10_songs.write.format("delta").mode("overwrite") \
    .save("/data/warehouse/delta/lastfm_top10_songs")

print("Results written to Delta.")

# Quick verification
spark.read.format("delta").load("/data/warehouse/delta/lastfm_top10_songs").show(truncate=60)

## Bonus — Summary statistics

In [ ]:
total_sessions = session_sizes.count()
total_plays    = plays.count()
unique_users   = plays.select("user_id").distinct().count()

print(f"Total users   : {unique_users:>10,}")
print(f"Total plays   : {total_plays:>10,}")
print(f"Total sessions: {total_sessions:>10,}")
print(f"Avg plays/session: {total_plays / total_sessions:.1f}")

session_sizes.agg(
    F.min("track_count").alias("min_session"),
    F.avg("track_count").alias("avg_session"),
    F.max("track_count").alias("max_session"),
).show()